In [1]:
import random
import numpy as np

random.seed(189)
np.random.seed(189)

from sklearn import svm
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from scipy import io
import pandas as pd

In [2]:
mnist = np.load("/kaggle/input/mnist-data/mnist-data.npz") # May have to change file path
mnist_training_data, mnist_training_labels = mnist['training_data'], mnist['training_labels']
mnist_test_data = mnist['test_data']

In [3]:
# Function that shuffles data given a dataset and its labels
def shuffle_data_labels(data, labels):
    data_points = np.arange(data.shape[0])
    np.random.shuffle(data_points)
    return [data[data_points], labels[data_points]]
    
# Shuffle dataset
mnist_training_data, mnist_training_labels = shuffle_data_labels(mnist_training_data, mnist_training_labels)

# Partition the shuffled datasets
mnist_training_set_data, mnist_training_set_labels = mnist_training_data[10000:, :], mnist_training_labels[10000:]
mnist_validation_set_data, mnist_validation_set_labels = mnist_training_data[:10000, :], mnist_training_labels[:10000]

# Reduce training size
subset_size = 20000
mnist_training_subset_data, mnist_training_subset_labels = mnist_training_set_data[:subset_size], mnist_training_set_labels[:subset_size]

# Flatten training/valdiation/test set data
mnist_training_subset_data_flattened = mnist_training_subset_data.reshape(mnist_training_subset_data.shape[0], -1)
mnist_validation_set_data_flattened = mnist_validation_set_data.reshape(mnist_validation_set_data.shape[0], -1)
mnist_test_data_flattened = mnist_test_data.reshape(mnist_test_data.shape[0], -1)

In [5]:
# Standardize the data
scaler = StandardScaler()
mnist_training_subset_data_flattened_scaled = scaler.fit_transform(mnist_training_subset_data_flattened)
mnist_validation_set_data_flattened_scaled = scaler.transform(mnist_validation_set_data_flattened)
mnist_test_data_flattened_scaled = scaler.transform(mnist_test_data_flattened)

In [6]:
# Train model
model = SVC(kernel='rbf', C=1, max_iter=1000000)
model.fit(mnist_training_subset_data_flattened_scaled, mnist_training_subset_labels)

SVC(C=1, max_iter=1000000)

In [7]:
# Evaluate model
training_accuracy = model.score(mnist_training_subset_data_flattened_scaled, mnist_training_subset_labels)
validation_accuracy = model.score(mnist_validation_set_data_flattened_scaled, mnist_validation_set_labels)
print(f"Training Accuracy: {training_accuracy}")
print(f"Validation Accuracy: {validation_accuracy}")

Training Accuracy: 0.9851
Validation Accuracy: 0.9527


In [8]:
# Generate CSV file
def results_to_csv(y_test, file_name):
    y_test = y_test.astype(int)
    df = pd.DataFrame({'Category': y_test})
    df.index += 1
    df.to_csv(file_name, index_label='Id')

In [9]:
y_test = model.predict(mnist_test_data_flattened_scaled)
results_to_csv(y_test, 'mnist-submission.csv')
print("Submission file 'mnist-submission.csv' has been created.")

Submission file 'mnist-submission.csv' has been created.
